# TASK 2

In [1]:
# Run this cell first
# Relevant imports
# import pandas as pd
import geopandas as gpd
import requests
import psycopg2
import json
from shapely.ops import unary_union

# == DB START UP ==
# Load json config
with open('../credentials.json', 'r') as f:
    config = json.load(f)

# Connection
conn = psycopg2.connect(**config)
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")
conn.commit()

In [2]:
# DB TABLE CREATION - Run only once
sa4_regions = ['Central Coast', 'Sydney - Blacktown', 'Sydney - Ryde', 'Sydney - Inner West']

def zone_to_table(zone):
    return f"{zone.lower().replace(' - ', '_').replace(' ', '_')}"

# CREATE ALL TABLES
for zone in sa4_regions:
    table_name = zone_to_table(zone)

    cur.execute(f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
            id              SERIAL PRIMARY KEY,
            sa2_name        TEXT,
            poi_id          TEXT UNIQUE,
            poi_name        TEXT,
            poi_category    TEXT,
            poi_type        INTEGER,
            geom            GEOMETRY(Point, 4326)
        );
    """)

    print(f'Created table: {table_name}')

conn.commit()

Created table: central_coast
Created table: sydney_blacktown
Created table: sydney_ryde
Created table: sydney_inner_west


In [3]:
#rather than a single POI, what if we want to find all POIs centered around a given POI?
#boxsize = 5 means a 5km by 5km region of interest centered around the midpoint (POI)

def nearbyPOI(bbox, filters={}):
    if (len(bbox) != 4):
        return "Invalid bounding box. Please provide a tuple of (xmin, ymin, xmax, ymax)."

    baseURL = 'https://maps.six.nsw.gov.au/arcgis/rest/services/public/NSW_POI/MapServer/0/query'
    xmin = round(bbox[0],5)
    ymin = round(bbox[1],5)
    xmax = round(bbox[2],5)
    ymax = round(bbox[3],5)

    params = {
        'geometry': f'"xmin":{xmin},"ymin":{ymin},"xmax":{xmax},"ymax":{ymax}',
        'outFields': '*',
        'returnGeometry': 'true',
        'f': 'json'
    }
    response = requests.get(baseURL, params)
    return response.json()['features']

## Individual SA4

## Inner West

In [4]:
#Load and filter SA2 data
sa2= gpd.read_file('../data/SA2_data/SA2_2021_AUST_GDA2020.shp')
sa4_name= 'Sydney - Inner West'

innerwest= sa2[sa2['SA4_NAME21']==sa4_name]

#check all SA2s within SA4 boundary
sa4_boundary= unary_union(innerwest.geometry)
print(f"\nVerifying {len(innerwest)} SA2s within '{sa4_name}' SA4 boundary:\n")

all_valid = True
for idx, row in innerwest.iterrows():
    is_within = row.geometry.within(sa4_boundary)
    status = "✓" if is_within else "✗"
    print(f"  {status} {row['SA2_NAME21']}")
    if not is_within:
        all_valid = False

print(f"Total SA2s: {len(innerwest)}")
print("All SA2s within SA4 boundary:", all_valid)

innerwest_data_dict={}

for index,row in innerwest.iterrows():
    innerwest_data_dict[row['SA2_NAME21']]=nearbyPOI(row.geometry.bounds)

print(f"SA2s processed. Ready for ingest.")



Verifying 21 SA2s within 'Sydney - Inner West' SA4 boundary:

  ✓ Concord - Mortlake - Cabarita
  ✓ Drummoyne - Rodd Point
  ✓ Five Dock - Abbotsford
  ✓ Concord West - North Strathfield
  ✓ Rhodes
  ✓ Balmain
  ✓ Lilyfield - Rozelle
  ✓ Annandale (NSW)
  ✓ Leichhardt
  ✓ Canterbury (North) - Ashbury
  ✓ Croydon Park - Enfield
  ✓ Dulwich Hill - Lewisham
  ✓ Haberfield - Summer Hill
  ✓ Homebush
  ✓ Strathfield South
  ✓ Ashfield - North
  ✓ Ashfield - South
  ✓ Burwood (NSW)
  ✓ Croydon
  ✓ Strathfield - East
  ✓ Strathfield - West
Total SA2s: 21
All SA2s within SA4 boundary: True
SA2s processed. Ready for ingest.


In [5]:
# === DB INGEST ===
cur.execute(f"TRUNCATE TABLE sydney_inner_west RESTART IDENTITY;")
conn.commit()

rows=[]
for sa2_name, features in innerwest_data_dict.items():
    for feat in features:
        props= feat.get('attributes', {})
        geom= feat.get('geometry', {})

        longitude=geom.get('x')
        latitude=geom.get('y')


        if longitude is None or latitude is None:
            continue

        rows.append((
            sa2_name,
            props.get('objectid'),
            props.get('poiname'),
            props.get('poitype'),
            int(props.get('poigroup')),
            longitude,
            latitude
        ))

cur.executemany(f"""
    INSERT INTO sydney_inner_west
    (sa2_name,
     poi_id,
     poi_name,
     poi_category,
     poi_type,
     geom)
    VALUES
    (%s, %s, %s, %s, %s,
     ST_SetSRID(ST_MakePoint(%s, %s), 4326))
     ON CONFLICT (poi_id) DO NOTHING
    """, rows)

conn.commit()
print(f"Ingestion complete. {len(rows)} POIs inserted into sydney_inner_west")

Ingestion complete. 3462 POIs inserted into sydney_inner_west


In [6]:
cur.close()
conn.close()